In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost
import os
import seaborn as sns

In [2]:
df = pd.read_csv('E:\\fraud detection\\data\\raw\\Synthetic_Financial_datasets_log2.csv')
df.head()

,tx_id,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,tx_id_1,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,tx_id_2,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,tx_id_3,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,tx_id_4,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,tx_id_5,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
df.drop(['isFlaggedFraud'], axis=1, inplace=True)
df.head(2)

,tx_id,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,tx_id_1,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0
1,tx_id_2,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0


In [4]:
for col in df.columns:
    print(f"{col}: {df[col].isnull().sum()}")

tx_id: 0
step: 0
type: 0
amount: 0
nameOrig: 0
oldbalanceOrg: 0
newbalanceOrig: 0
nameDest: 0
oldbalanceDest: 0
newbalanceDest: 0
isFraud: 0


In [5]:
df.shape

(1048575, 11)

In [6]:
df.dtypes

tx_id              object
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
dtype: object

In [7]:
print(df.isnull().sum().sort_values(ascending=False).head(10))

tx_id             0
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
dtype: int64


In [8]:
print(df.duplicated().sum())

0


In [9]:
df["isFraud"].value_counts()

isFraud
0    1047433
1       1142
Name: count, dtype: int64

In [10]:
df["isFraud"].value_counts(normalize=True)

isFraud
0    0.998911
1    0.001089
Name: proportion, dtype: float64

In [11]:
for col in df.columns:
    print(f"{col}: {df[col].isna().sum()}")

tx_id: 0
step: 0
type: 0
amount: 0
nameOrig: 0
oldbalanceOrg: 0
newbalanceOrig: 0
nameDest: 0
oldbalanceDest: 0
newbalanceDest: 0
isFraud: 0


In [12]:
df.groupby("type")["isFraud"].agg(["count", "sum", "mean"]).sort_values("mean", ascending=False)

,count,sum,mean
type,,,
TRANSFER,86753,564,0.006501
CASH_OUT,373641,578,0.001547
CASH_IN,227130,0,0.000000
DEBIT,7178,0,0.000000
PAYMENT,353873,0,0.000000


In [13]:
df.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,1047433.0,1.575397e+05,2.541883e+05,0.1,12134.87,76214.97,2.134928e+05,6419835.27
1,1142.0,1.192629e+06,2.030599e+06,119.0,86070.17,353179.45,1.248759e+06,10000000.00


In [14]:
df.drop("step", axis=1, inplace=True)
df.head(1)

,tx_id,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,tx_id_1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0


In [16]:


from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("E:/fraud detection/data/raw/Synthetic_Financial_datasets_log2.csv")
OUTPUT_DIR = Path("data/processed")

TARGET = "isFraud"


def main():
    df = pd.read_csv(DATA_PATH)

    if TARGET not in df.columns:
        raise ValueError(f"Missing target column: {TARGET}")

    if df[TARGET].isna().any():
        raise ValueError("Target contains missing values.")

    # 80% training, 20% temporary data
    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42,
        stratify=df[TARGET],
    )

    # Split remaining 20% equally: 10% validation, 10% testing
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        stratify=temp_df[TARGET],
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    splits = {
        "train": train_df,
        "validation": val_df,
        "test": test_df,
    }

    for name, part in splits.items():
        part.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

        print(f"\n--- {name.upper()} ---")
        print(f"Rows: {len(part):,} ({len(part) / len(df):.2%})")
        print("Class counts:")
        print(part[TARGET].value_counts())
        print("Fraud rate:", f"{part[TARGET].mean():.5%}")

    print("\nAll splits saved to:", OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()



--- TRAIN ---
Rows: 838,860 (80.00%)
Class counts:
isFraud
0    837946
1       914
Name: count, dtype: int64
Fraud rate: 0.10896%

--- VALIDATION ---
Rows: 104,857 (10.00%)
Class counts:
isFraud
0    104743
1       114
Name: count, dtype: int64
Fraud rate: 0.10872%

--- TEST ---
Rows: 104,858 (10.00%)
Class counts:
isFraud
0    104744
1       114
Name: count, dtype: int64
Fraud rate: 0.10872%

All splits saved to: E:\fraud detection\data\processed


In [14]:
train_df = pd.read_csv(OUTPUT_DIR / "train.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test.csv")
val_df = pd.read_csv(OUTPUT_DIR / "validation.csv")

TARGET = "isFraud"

EXCLUDED_COLUMNS = [
    "isFraud",
    "step",
    "nameOrig",
    "nameDest",
    "isFlaggedFraud",
]

feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDED_COLUMNS
]

X_train = train_df[feature_cols]
y_train = train_df[TARGET]

X_val = val_df[feature_cols]
y_val = val_df[TARGET]

X_test = test_df[feature_cols]
y_test = test_df[TARGET]

In [15]:
X_train.head()

,tx_id,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
0,tx_id_70607,PAYMENT,2265.26,20950.0,18684.74,0.00,0.00
1,tx_id_701855,CASH_IN,364162.66,50066.0,414228.66,0.00,0.00
2,tx_id_261359,PAYMENT,4338.74,0.0,0.00,0.00,0.00
3,tx_id_412027,CASH_IN,299163.26,820.0,299983.26,0.00,0.00
4,tx_id_897617,CASH_IN,25652.02,3522.0,29174.02,219292.18,193640.17


In [16]:

from pathlib import Path
import json

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from xgboost import XGBClassifier


DATA_DIR = Path("data/processed")
MODEL_DIR = Path("models")
REPORT_DIR = Path("reports")

TARGET = "isFraud"

EXCLUDED_COLUMNS = {
    "isFraud",
    "step",
    "nameOrig",
    "nameDest",
    "isFlaggedFraud",
    "tx_id"
}


def main():
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    val_df = pd.read_csv(DATA_DIR / "validation.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")

    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDED_COLUMNS
    ]

    X_train = train_df[feature_cols]
    y_train = train_df[TARGET].astype(int)

    X_val = val_df[feature_cols]
    y_val = val_df[TARGET].astype(int)

    X_test = test_df[feature_cols]
    y_test = test_df[TARGET].astype(int)

    categorical = X_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numerical = [
        col for col in feature_cols
        if col not in categorical
    ]

    preprocessor = ColumnTransformer([
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numerical,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical,
        ),
    ])

    # Handle class imbalance using training data only.
    positives = int(y_train.sum())
    negatives = len(y_train) - positives

    if positives == 0:
        raise ValueError("Training set contains no fraud examples.")

    scale_pos_weight = negatives / positives

    models = {
        "logistic_regression": LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
        ),
        "xgboost": XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="aucpr",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        ),
    }

    results = {}
    trained_models = {}

    for name, classifier in models.items():
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", classifier),
        ])

        pipeline.fit(X_train, y_train)

        probabilities = pipeline.predict_proba(X_val)[:, 1]

        # Initial threshold only; we will tune it later.
        predictions = (probabilities >= 0.7).astype(int)

        metrics = {
            "average_precision": float(
                average_precision_score(y_val, probabilities)
            ),
            "precision": float(
                precision_score(
                    y_val, predictions, zero_division=0
                )
            ),
            "recall": float(
                recall_score(
                    y_val, predictions, zero_division=0
                )
            ),
            "f1": float(
                f1_score(y_val, predictions, zero_division=0)
            ),
        }

        results[name] = metrics
        trained_models[name] = pipeline

        print(f"\n{name.upper()}")
        print(json.dumps(metrics, indent=2))

    # Select using validation average precision.
    best_name = max(
        results,
        key=lambda name: results[name]["average_precision"],
    )

    best_model = trained_models[best_name]

    # Final evaluation on the untouched test split.
    test_probabilities = best_model.predict_proba(X_test)[:, 1]
    test_predictions = (
        test_probabilities >= 0.7
    ).astype(int)

    test_metrics = {
        "selected_model": best_name,
        "threshold": 0.7,
        "average_precision": float(
            average_precision_score(y_test, test_probabilities)
        ),
        "precision": float(
            precision_score(
                y_test, test_predictions, zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                y_test, test_predictions, zero_division=0
            )
        ),
        "f1": float(
            f1_score(y_test, test_predictions, zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            y_test, test_predictions
        ).tolist(),
    }

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    joblib.dump(
        best_model,
        MODEL_DIR / "baseline_fraud_model.joblib",
    )

    with open(
        REPORT_DIR / "validation_metrics.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(results, file, indent=2)

    with open(
        REPORT_DIR / "test_metrics.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(test_metrics, file, indent=2)

    print("\nFINAL TEST RESULTS")
    print(json.dumps(test_metrics, indent=2))
    print("\nSaved model:", MODEL_DIR / "baseline_fraud_model.joblib")


if __name__ == "__main__":
    main()



LOGISTIC_REGRESSION
{
  "average_precision": 0.4311408457334638,
  "precision": 0.038781163434903045,
  "recall": 0.8596491228070176,
  "f1": 0.07421431276031806
}

XGBOOST
{
  "average_precision": 0.8525983898265524,
  "precision": 0.49514563106796117,
  "recall": 0.8947368421052632,
  "f1": 0.6375
}

FINAL TEST RESULTS
{
  "selected_model": "xgboost",
  "threshold": 0.7,
  "average_precision": 0.842933146114887,
  "precision": 0.5414364640883977,
  "recall": 0.8596491228070176,
  "f1": 0.6644067796610169,
  "confusion_matrix": [
    [
      104661,
      83
    ],
    [
      16,
      98
    ]
  ]
}

Saved model: models\baseline_fraud_model.joblib
